# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imtiyazsoomro/flyrank-ml-internship-imtiyaz/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue must translate raw probabilities into actions a human trusts. We sort the backlog first by the model's confidence that the page will decline, and secondly by the page's current visibility (impressions_90d).

Reason Codes:

URGENT_REFRESH: High probability of decline (>60%) + High current visibility (>1000 impressions). These pages have the most to lose.

STANDARD_REFRESH: High probability of decline (>60%) + Low visibility.

MONITOR: Low probability of decline. Leave these pages alone.

In [ ]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 1. Setup environment and load data
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/imtiyazsoomro/flyrank-ml-internship-imtiyaz.git
    %cd flyrank-ml-internship-imtiyaz

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# 2. Train the model to generate probabilities
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'word_count']
X = df[features].fillna(df[features].median())
y = df['is_declining']

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X, y)

# 3. Predict probability of decline
df['decline_probability'] = rf.predict_proba(X)[:, 1]

# 4. Map actions and reason codes
def assign_playbook_action(row):
    if row['decline_probability'] > 0.6 and row['impressions_90d'] > 1000:
        return 'URGENT_REFRESH', 'HIGH_RISK_HIGH_VALUE'
    elif row['decline_probability'] > 0.6:
        return 'STANDARD_REFRESH', 'HIGH_RISK_LOW_VALUE'
    else:
        return 'MONITOR', 'STABLE'

df[['action', 'reason_code']] = df.apply(lambda r: pd.Series(assign_playbook_action(r)), axis=1)

# 5. Build the ranked queue
playbook_queue = df.sort_values(by=['decline_probability', 'impressions_90d'], ascending=[False, False])

print("=== CONTENT ACTION PLAYBOOK (TOP 5) ===")
print(playbook_queue[['decline_probability', 'impressions_90d', 'action', 'reason_code']].head(5).to_string(index=False))

Cloning into 'flyrank-ml-internship-imtiyaz'...
remote: Enumerating objects: 204, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 204 (delta 100), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (204/204), 1.87 MiB | 837.00 KiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/flyrank-ml-internship-imtiyaz
=== CONTENT ACTION PLAYBOOK (TOP 5) ===
 decline_probability  impressions_90d         action          reason_code
            0.914795            48471 URGENT_REFRESH HIGH_RISK_HIGH_VALUE
            0.906905             3412 URGENT_REFRESH HIGH_RISK_HIGH_VALUE
            0.905479             5454 URGENT_REFRESH HIGH_RISK_HIGH_VALUE
            0.898855             1531 URGENT_REFRESH HIGH_RISK_HIGH_VALUE
            0.898786            34089 URGENT_REFRESH HIGH_RISK_HIGH_VALUE


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use: This playbook is designed as a decision-support tool for editorial teams. It highlights which pages are statistically likely to lose traffic soon, helping editors prioritize their manual refresh cycles to protect existing search visibility.

Limits: The model observes symptoms of decay (age, staleness, dropping ranks), not the root cause. It cannot tell an editor if a page is dropping due to a technical SEO error, a new competitor, or outdated facts. It stops being valid if applied to entirely new content types (e.g., short-form news or video pages) that were not in the training set.

In [ ]:
# Print out limits to ensure they are tracked alongside the data
print("LIMITATION WARNING: This queue flags symptomatic decay probability. It does not diagnose technical SEO or intent-mismatch issues.")

LIMITATION WARNING: This queue flags symptomatic decay probability. It does not diagnose technical SEO or intent-mismatch issues.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

What a person must check:
A human editor must review URGENT_REFRESH pages to verify why the page is aging poorly. They must check if the content simply needs updated statistics, or if the search intent for the keyword has fundamentally changed.

The No-Go List (Never Automate):

Auto-deleting content: Never allow a script to automatically unpublish or delete a page just because its decline probability is high.

Auto-rewriting with Generative AI: Do not hook this queue directly into an LLM to blindly rewrite the text. High-value pages require human subject-matter expertise to update accurately.

Touching Brand-New Content: Content younger than 30 days should be completely excluded from the action queue, as its metrics are too volatile.

In [ ]:
# Apply the No-Go rule: Filter out brand new content from taking action
new_content_mask = playbook_queue['content_age_days'] <= 30
playbook_queue.loc[new_content_mask, 'action'] = 'NO_TOUCH'
playbook_queue.loc[new_content_mask, 'reason_code'] = 'CONTENT_TOO_NEW'

print(f"Filtered {new_content_mask.sum()} brand new pages into the NO_TOUCH bucket to prevent automated mistakes.")

Filtered 0 brand new pages into the NO_TOUCH bucket to prevent automated mistakes.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

To ensure the recommendations do not go stale, the model must be monitored and retrained based on the following triggers:

Data Drift: Retrain if the median content_age_days or impressions_90d of the active portfolio shifts by more than 20% month-over-month.

Macro SEO Events: Retrain the model immediately following a confirmed Google Core Algorithm Update, as the rules of visibility often change fundamentally.

Performance Drop: Re-evaluate if the model's precision on a newly captured 30-day holdout set drops below 65%.

In [ ]:
# Establish baseline metrics for future drift monitoring
monitoring_baseline = {
    'median_age': playbook_queue['content_age_days'].median(),
    'median_impressions': playbook_queue['impressions_90d'].median()
}

print("=== DRIFT MONITORING BASELINES ESTABLISHED ===")
print(f"Median Age: {monitoring_baseline['median_age']} days")
print(f"Median Impressions: {monitoring_baseline['median_impressions']}")

=== DRIFT MONITORING BASELINES ESTABLISHED ===
Median Age: 236.0 days
Median Impressions: 731.0


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# Create output directory
os.makedirs('work/outputs', exist_ok=True)

# Select final columns for the playbook export (removed missing content_hash_id)
export_cols = [
    'decline_probability',
    'action',
    'reason_code',
    'content_age_days',
    'days_since_last_update',
    'impressions_90d'
]

# Save the artifact
output_csv_path = 'work/outputs/action_playbook_queue.csv'
playbook_queue[export_cols].to_csv(output_csv_path, index=False)

print(f"Success! Action Playbook exported to: {output_csv_path}")
print(f"Total actionable rows: {len(playbook_queue)}")

Success! Action Playbook exported to: work/outputs/action_playbook_queue.csv
Total actionable rows: 30000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.